# R notebook for regression models and figure export

This notebook is split into separate cells and does the following:

- prepares the 1997–2023 dataset and saves it to CSV
- runs regression equation (5)
- runs regression equation (6) based on principal components
- prepares the standardized datasets used for the coefficient variation diagrams
- saves all figure outputs as **PDF** files using `cairo_pdf(..., fallback_resolution = 300)`
- saves key text outputs to `.txt` files

All generated files are written into these folders relative to the notebook location:

- `data/`
- `outputs/`


In [1]:
# Setup ----------------------------------------------------------------------

dir.create("data", showWarnings = FALSE)
dir.create("outputs", showWarnings = FALSE)

install_if_missing <- function(pkgs) {
  to_install <- pkgs[!pkgs %in% rownames(installed.packages())]
  if (length(to_install) > 0) {
    install.packages(to_install, repos = "https://cloud.r-project.org")
  }
}

install_if_missing(c("lars", "glmnet"))

library(lars)
library(glmnet)

save_plot_pdf <- function(filename, plot_expr, width = 7, height = 6) {
  grDevices::cairo_pdf(
    filename = file.path("outputs", filename),
    width = width,
    height = height,
    fallback_resolution = 300
  )
  on.exit(dev.off(), add = TRUE)
  eval.parent(substitute(plot_expr))
}

standardize_for_path <- function(v) {
  v <- v - mean(v)
  k <- length(v) / sqrt(drop(crossprod(v)))
  v * k
}

Loaded lars 1.3


Warning message:
"package 'glmnet' was built under R version 4.4.3"
Loading required package: Matrix

Loaded glmnet 4.1-10



## 1. Regression equation (5): prepare dataset and save CSV

In [2]:
# Regression equation (5) data ------------------------------------------------

year <- 1997:2023

# 1997-2023 price of commercial housing
y <- c(1997,2063,2053,2112,2170,2250,2359,2714,3168,3367,3864,3800,4681,5045,5384,5839,6328,6427,6932,7699,8160,9045,9673,10248,10546,10210,10438) / 1997

# 1997-2023 Chinese gross domestic product
x1 <- c(79715.0,85195.5,90564.4,100280.1,110863.1,121717.4,137422.0,161840.2,187318.9,219438.5,270092.3,319244.6,348517.7,412119.3,487940.2,538580.0,592963.2,643563.1,688858.2,746395.1,832035.9,919281.1,986515.2,1013567.0,1149237.0,1204724.0,1260582.1) / 79715.0

# 1997-2023 Chinese fiscal revenue
x2 <- c(8651.14,9875.95,11444.08,13395.23,16386.04,18903.64,21715.25,26396.47,31649.29,38760.20,51321.78,61330.35,68518.30,83101.51,103874.43,117253.52,129209.64,140370.03,152269.23,159269.97,172592.77,183359.84,190390.08,182913.88,202554.64,203649.29,216795.43) / 8651.14

# 1997-2023 Chinese broad money supply
x3 <- c(90995.3,104498.5,119897.9,134610.3,158301.9,185007.0,221222.8,254107.0,298755.7,345577.9,403442.2,475166.6,610224.5,725851.8,851590.9,974148.8,1106525.0,1228374.8,1392278.1,1550066.7,1690235.3,1826744.2,1986488.8,2186795.9,2382899.6,2664320.8,2922713.3) / 90995.3

# 1997-2023 Chinese consumer price index
x4 <- c(100,99.2,98.6,100.4,100.7,99.2,101.2,103.9,101.8,101.5,104.8,105.9,99.3,103.3,105.4,102.6,102.6,102.0,101.4,102.0,101.6,102.1,102.9,102.5,100.9,102.0,100.2) / 100
for (i in 2:27) {
  x4[i] <- x4[i - 1] * x4[i]
}

# 1997-2023 fixed assets investment of the whole society in China
x5 <- c(24941,28406,29855,32918,37214,43500,53841,66235,80994,97583,118323,144587,181760,218834,205036,241746,282486,320331,347827,372021,394926,418215,439541,451155,473003,495966,509708) / 24941

# 1997-2023 the cost of commercial housing in China
x6 <- c(1175,1218,1152,1139,1128,1184,1273,1402,1451,1564,1657,1795,2021,2228,2373,2498,2643,2816,3054,3039,3105,3210,3549,3781,3891,4093,4150) / 1175

# 1997-2023 per capita disposable income of urban residents in China
x7 <- c(5160.3,5425.1,5854.0,6280.0,6859.6,7702.8,8472.2,9421.6,10493.0,11759.5,13785.8,15780.8,17174.7,19109.4,21809.8,24564.7,26955.1,28843.9,31194.8,33616.2,36396.2,39250.8,42358.8,43833.8,47412,49283,51821) / 5160.3

X <- data.frame(year, x1, x2, x3, x4, x5, x6, x7, y)

write.csv(X, file = file.path("data", "regression_data_1997_2023.csv"), row.names = FALSE)

X

year,x1,x2,x3,x4,x5,x6,x7,y
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1997,1.000000,1.000000,1.000000,1.0000000,1.000000,1.0000000,1.000000,1.000000
1998,1.068751,1.141578,1.148394,0.9920000,1.138928,1.0365957,1.051315,1.033050
1999,1.136102,1.322841,1.317627,0.9781120,1.197025,0.9804255,1.134430,1.028042
2000,1.257983,1.548377,1.479310,0.9820244,1.319835,0.9693617,1.216984,1.057586
2001,1.390743,1.894090,1.739671,0.9888986,1.492081,0.9600000,1.329303,1.086630
2002,1.526907,2.185104,2.033149,0.9809874,1.744116,1.0076596,1.492704,1.126690
2003,1.723916,2.510103,2.431145,0.9927593,2.158735,1.0834043,1.641804,1.181272
2004,2.030235,3.051213,2.792529,1.0314769,2.655667,1.1931915,1.825785,1.359039
2005,2.349858,3.658395,3.283199,1.0500435,3.247424,1.2348936,2.033409,1.586380


## 2. Fit regression equation (5) and save summary

In [3]:
# Regression equation (5) -----------------------------------------------------

hp <- lm(y ~ x1 + x2 + x3 + x4 + x5 + x6 + x7, data = X)

summary_hp <- summary(hp)
summary_hp

capture.output(summary_hp, file = file.path("outputs", "regression_equation_5_summary.txt"))


Call:
lm(formula = y ~ x1 + x2 + x3 + x4 + x5 + x6 + x7, data = X)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.26512 -0.05632  0.01297  0.05291  0.23561 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)   
(Intercept)  0.64371    1.53949   0.418  0.68054   
x1           0.11011    0.18589   0.592  0.56062   
x2          -0.08653    0.04368  -1.981  0.06223 . 
x3          -0.13567    0.04616  -2.939  0.00842 **
x4          -0.78316    1.94359  -0.403  0.69149   
x5           0.07412    0.06338   1.169  0.25668   
x6           0.23589    0.49223   0.479  0.63725   
x7           0.83106    0.33954   2.448  0.02427 * 
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.1206 on 19 degrees of freedom
Multiple R-squared:  0.9955,	Adjusted R-squared:  0.9938 
F-statistic: 596.3 on 7 and 19 DF,  p-value: < 2.2e-16


## 3. Fit regression equation (6) via principal components and save summary

In [4]:
# Regression equation (6) -----------------------------------------------------

hp.pr <- princomp(~ x1 + x2 + x3 + x4 + x5 + x6 + x7, data = X, cor = TRUE)
summary_hp_pr <- summary(hp.pr, loadings = TRUE)
summary_hp_pr

pre <- predict(hp.pr)
X$z1 <- pre[, 1]

lm.sol <- lm(y ~ z1, data = X)
summary_lm_sol <- summary(lm.sol)
summary_lm_sol

beta <- coef(lm.sol)
A <- loadings(hp.pr)
x.bar <- hp.pr$center
x.sd <- hp.pr$scale
coef_pc <- beta[2] * A[, 1] / x.sd
beta0 <- beta[1] - sum(x.bar * coef_pc)

pc_regression_coefficients <- c(beta0 = beta0, coef_pc)
pc_regression_coefficients

capture.output(summary_hp_pr, file = file.path("outputs", "regression_equation_6_pca_summary.txt"))
capture.output(summary_lm_sol, file = file.path("outputs", "regression_equation_6_lm_summary.txt"))
write.csv(
  data.frame(term = names(pc_regression_coefficients), value = as.numeric(pc_regression_coefficients)),
  file = file.path("outputs", "regression_equation_6_coefficients.csv"),
  row.names = FALSE
)

Importance of components:
                          Comp.1      Comp.2      Comp.3       Comp.4
Standard deviation     2.6349591 0.206567137 0.085060511 0.0620737031
Proportion of Variance 0.9918585 0.006095712 0.001033613 0.0005504492
Cumulative Proportion  0.9918585 0.997954172 0.998987785 0.9995382345
                             Comp.5       Comp.6       Comp.7
Standard deviation     0.0428412881 0.0326148374 0.0182552614
Proportion of Variance 0.0002621966 0.0001519611 0.0000476078
Cumulative Proportion  0.9998004311 0.9999523922 1.0000000000

Loadings:
   Comp.1 Comp.2 Comp.3 Comp.4 Comp.5 Comp.6 Comp.7
x1  0.379  0.309         0.339  0.293  0.370  0.647
x2  0.377 -0.435  0.512  0.251 -0.578              
x3  0.376  0.674               -0.328 -0.536       
x4  0.378 -0.414 -0.464  0.409  0.245 -0.493       
x5  0.378 -0.249  0.315 -0.697  0.381 -0.193  0.166
x6  0.379        -0.629 -0.382 -0.374  0.417       
x7  0.379  0.162  0.144  0.138  0.357  0.336 -0.741


Call:
lm(formula = y ~ z1, data = X)

Residuals:
     Min       1Q   Median       3Q      Max 
-0.29860 -0.05156 -0.01894  0.10229  0.34340 

Coefficients:
            Estimate Std. Error t value Pr(>|t|)    
(Intercept)  2.75547    0.02894   95.21   <2e-16 ***
z1           0.56788    0.01098   51.70   <2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Residual standard error: 0.1504 on 25 degrees of freedom
Multiple R-squared:  0.9907,	Adjusted R-squared:  0.9904 
F-statistic:  2673 on 1 and 25 DF,  p-value: < 2.2e-16


beta0.(Intercept)                x1                x2                x3 
      -0.38999198        0.04498709        0.02562115        0.02259855 
               x4                x5                x6                x7 
       0.97450980        0.03190381        0.25029377        0.07354322

## 4. Prepare standardized data for coefficient variation diagrams (Figures 1–3)

In [5]:
# Standardized data for coefficient variation diagrams 1, 2, 3 ---------------

y_std <- c(1997,2063,2053,2112,2170,2250,2359,2714,3168,3367,3864,3800,4681,5045,5384,5839,6328,6427,6932,7699,8160,9045,9673,10248,10546,10210,10438) / 1997
y_std <- y_std - mean(y_std)

x1_std <- c(79715.0,85195.5,90564.4,100280.1,110863.1,121717.4,137422.0,161840.2,187318.9,219438.5,270092.3,319244.6,348517.7,412119.3,487940.2,538580.0,592963.2,643563.1,688858.2,746395.1,832035.9,919281.1,986515.2,1013567.0,1149237.0,1204724.0,1260582.1) / 79715.0
x1_std <- standardize_for_path(x1_std)

x2_std <- c(8651.14,9875.95,11444.08,13395.23,16386.04,18903.64,21715.25,26396.47,31649.29,38760.20,51321.78,61330.35,68518.30,83101.51,103874.43,117253.52,129209.64,140370.03,152269.23,159269.97,172592.77,183359.84,190390.08,182913.88,202554.64,203649.29,216795.43) / 8651.14
x2_std <- standardize_for_path(x2_std)

x3_std <- c(90995.3,104498.5,119897.9,134610.3,158301.9,185007.0,221222.8,254107.0,298755.7,345577.9,403442.2,475166.6,610224.5,725851.8,851590.9,974148.8,1106525.0,1228374.8,1392278.1,1550066.7,1690235.3,1826744.2,1986488.8,2186795.9,2382899.6,2664320.8,2922713.3) / 90995.3
x3_std <- standardize_for_path(x3_std)

x4_std <- c(100,99.2,98.6,100.4,100.7,99.2,101.2,103.9,101.8,101.5,104.8,105.9,99.3,103.3,105.4,102.6,102.6,102.0,101.4,102.0,101.6,102.1,102.9,102.5,100.9,102.0,100.2) / 100
for (i in 2:27) {
  x4_std[i] <- x4_std[i - 1] * x4_std[i]
}
x4_std <- standardize_for_path(x4_std)

x5_std <- c(24941,28406,29855,32918,37214,43500,53841,66235,80994,97583,118323,144587,181760,218834,205036,241746,282486,320331,347827,372021,394926,418215,439541,451155,473003,495966,509708) / 24941
x5_std <- standardize_for_path(x5_std)

x6_std <- c(1175,1218,1152,1139,1128,1184,1273,1402,1451,1564,1657,1795,2021,2228,2373,2498,2643,2816,3054,3039,3105,3210,3549,3781,3891,4093,4150) / 1175
x6_std <- standardize_for_path(x6_std)

x7_std <- c(5160.3,5425.1,5854.0,6280.0,6859.6,7702.8,8472.2,9421.6,10493.0,11759.5,13785.8,15780.8,17174.7,19109.4,21809.8,24564.7,26955.1,28843.9,31194.8,33616.2,36396.2,39250.8,42358.8,43833.8,47412,49283,51821) / 5160.3
x7_std <- standardize_for_path(x7_std)

X_std <- cbind(x1_std, x2_std, x3_std, x4_std, x5_std, x6_std, x7_std)

standardized_data <- data.frame(
  year = 1997:2023,
  x1 = x1_std, x2 = x2_std, x3 = x3_std, x4 = x4_std,
  x5 = x5_std, x6 = x6_std, x7 = x7_std, y = y_std
)

write.csv(
  standardized_data,
  file = file.path("data", "standardized_data_1997_2023.csv"),
  row.names = FALSE
)

head(standardized_data)

,year,x1,x2,x3,x4,x5,x6,x7,y
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1997,-5.838348,-6.334849,-5.473377,-5.796556,-6.228377,-5.882467,-6.130846,-1.755467
2,1998,-5.763590,-6.246915,-5.391707,-5.985456,-6.121216,-5.661189,-6.039772,-1.722417
3,1999,-5.690355,-6.134332,-5.298569,-6.313385,-6.076403,-6.000826,-5.892258,-1.727424
4,2000,-5.557826,-5.994250,-5.209585,-6.221003,-5.981674,-6.067724,-5.745742,-1.697880
5,2001,-5.413467,-5.779527,-5.066294,-6.058687,-5.848812,-6.124330,-5.546397,-1.668837
6,2002,-5.265407,-5.598777,-4.904777,-6.245490,-5.654406,-5.836153,-5.256391,-1.628776


## 5. Generate and save Figures 1–3 as PDF

In [6]:
# Figures 1, 2, 3 ------------------------------------------------------------

c_lasso_1 <- lars(X_std, y_std, type = "lasso")
a_lar_1 <- lars(X_std, y_std, type = "lar")
b_glmnet_1 <- glmnet(X_std, y_std, alpha = 1, family = "gaussian")

save_plot_pdf("figure1_lasso_path_1997_2023.pdf", plot(c_lasso_1))
save_plot_pdf("figure2_lar_path_1997_2023.pdf", plot(a_lar_1))
save_plot_pdf("figure3_glmnet_path_1997_2023.pdf", plot(b_glmnet_1))

lar_beta_1 <- lars(X_std, y_std, type = "lar")$beta
lasso_beta_1 <- lars(X_std, y_std, type = "lasso")$beta
glmnet_beta_1 <- as.matrix(glmnet(X_std, y_std, alpha = 1, family = "gaussian")$beta)

lar_beta_1
lasso_beta_1
glmnet_beta_1

write.csv(lar_beta_1, file = file.path("outputs", "figure1_3_lar_beta.csv"), row.names = FALSE)
write.csv(lasso_beta_1, file = file.path("outputs", "figure1_3_lasso_beta.csv"), row.names = FALSE)
write.csv(glmnet_beta_1, file = file.path("outputs", "figure1_3_glmnet_beta.csv"), row.names = TRUE)

,x1_std,x2_std,x3_std,x4_std,x5_std,x6_std,x7_std
0,0.000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000
1,0.000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.09863443
2,0.000000,0.00000000,0.00000000,0.000000000,0.05725972,0.00000000,0.15589415
3,0.000000,0.00000000,0.00000000,0.032754300,0.06673289,0.00000000,0.18879449
4,0.000000,0.00000000,-0.05080159,0.009595910,0.04943748,0.00000000,0.27954915
5,0.000000,0.00000000,-0.07630591,-0.008246843,0.03994435,0.01455839,0.31756072
6,0.000000,-0.06060263,-0.14021239,-0.015086558,0.05682650,0.02396027,0.42182469
7,0.101259,-0.13931289,-0.24651985,-0.033167085,0.09609610,0.03901240,0.46825608


,x1_std,x2_std,x3_std,x4_std,x5_std,x6_std,x7_std
0,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000
1,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000000,0.09863443
2,0.00000000,0.00000000,0.00000000,0.00000000,0.05725972,0.000000000,0.15589415
3,0.00000000,0.00000000,0.00000000,0.03275430,0.06673289,0.000000000,0.18879449
4,0.00000000,0.00000000,-0.05080159,0.00959591,0.04943748,0.000000000,0.27954915
5,0.00000000,0.00000000,-0.06451792,0.00000000,0.04433203,0.007829564,0.29999194
6,0.00000000,0.00000000,-0.06928378,0.00000000,0.04064973,0.008314629,0.30792361
7,0.00000000,-0.07152128,-0.13783090,0.00000000,0.06126393,0.013299038,0.42153979
8,0.05933819,-0.12531396,-0.19845498,0.00000000,0.08739248,0.014632282,0.44854869
9,0.10125900,-0.13931289,-0.24651985,-0.03316708,0.09609610,0.039012401,0.46825608


,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,⋯,s45,s46,s47,s48,s49,s50,s51,s52,s53,s54
x1_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x2_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x3_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x4_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.02270462,0.02285529,0.02300565,0.02320507,0.02335521,0.02350544,0.02365572,0.02375635,0.02385708,0.02395792
x5_std,0,0.00000000,0.00000000,0.00000000,0.003777238,0.01262400,0.02072126,0.02804779,0.03471738,0.04080972,⋯,0.07728137,0.07735347,0.07739074,0.07730641,0.07728727,0.07724358,0.07717816,0.07718226,0.07716711,0.07713477
x6_std,0,0.00000000,0.00000000,0.00000000,0.000000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,⋯,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000
x7_std,0,0.02559004,0.04890673,0.07015203,0.085752905,0.09459165,0.10260895,0.10996514,0.11667386,0.12277143,⋯,0.18430913,0.18447705,0.18464537,0.18485453,0.18501901,0.18518169,0.18534210,0.18546087,0.18557891,0.18569593


## 6. Prepare data for Figures 4–6 and save CSV

In [7]:
# Figures 4, 5, 6 data --------------------------------------------------------

x1_f46 <- c(10,20,30,0,0,0,0,0,0,0,0,0,0,0,0,0)
x1_f46 <- standardize_for_path(x1_f46)

x2_f46 <- c(0,0,0,40,50,60,70,80,9,10,0,0,0,0,0,0)
x2_f46 <- standardize_for_path(x2_f46)

x3_f46 <- c(0,0,0,0,0,0,0,0,0,0,11,12,13,0,0,0)
x3_f46 <- standardize_for_path(x3_f46)

x4_f46 <- c(0,0,0,0,0,0,0,0,0,0,0,0,0,14,15,16)
x4_f46 <- standardize_for_path(x4_f46)

X_f46 <- cbind(x1_f46, x2_f46, x3_f46, x4_f46)
y_f46 <- c(-1,10,-5,-10,1,8,3,4,2,1,9,7,5,3,2,-1)
y_f46 <- y_f46 - mean(y_f46)

figure46_data <- data.frame(
  obs = 1:16,
  x1 = x1_f46, x2 = x2_f46, x3 = x3_f46, x4 = x4_f46, y = y_f46
)

write.csv(
  figure46_data,
  file = file.path("data", "figure4_6_data.csv"),
  row.names = FALSE
)

figure46_data

obs,x1,x2,x3,x4,y
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2.91730,-2.817285,-1.916087,-1.918044,-3.375
2,7.58498,-2.817285,-1.916087,-1.918044,7.625
3,12.25266,-2.817285,-1.916087,-1.918044,-7.375
4,-1.75038,2.834948,-1.916087,-1.918044,-12.375
5,-1.75038,4.248007,-1.916087,-1.918044,-1.375
6,-1.75038,5.661065,-1.916087,-1.918044,5.625
7,-1.75038,7.074123,-1.916087,-1.918044,0.625
8,-1.75038,8.487182,-1.916087,-1.918044,1.625
9,-1.75038,-1.545533,-1.916087,-1.918044,-0.375


## 7. Generate and save Figures 4–6 as PDF

In [8]:
# Figures 4, 5, 6 ------------------------------------------------------------

c_lasso_46 <- lars(X_f46, y_f46, type = "lasso")
a_lar_46 <- lars(X_f46, y_f46, type = "lar")
b_glmnet_46 <- glmnet(X_f46, y_f46, alpha = 1, family = "gaussian")

save_plot_pdf("figure4_lasso_path_block.pdf", plot(c_lasso_46))
save_plot_pdf("figure5_lar_path_block.pdf", plot(a_lar_46))
save_plot_pdf("figure6_glmnet_path_block.pdf", plot(b_glmnet_46))

lar_beta_46 <- lars(X_f46, y_f46, type = "lar")$beta
lasso_beta_46 <- lars(X_f46, y_f46, type = "lasso")$beta
glmnet_beta_46 <- as.matrix(glmnet(X_f46, y_f46, alpha = 1, family = "gaussian")$beta)

lar_beta_46
lasso_beta_46
glmnet_beta_46

write.csv(lar_beta_46, file = file.path("outputs", "figure4_6_lar_beta.csv"), row.names = FALSE)
write.csv(lasso_beta_46, file = file.path("outputs", "figure4_6_lasso_beta.csv"), row.names = FALSE)
write.csv(glmnet_beta_46, file = file.path("outputs", "figure4_6_glmnet_beta.csv"), row.names = TRUE)

,x1_f46,x2_f46,x3_f46,x4_f46
0,0.000000000,0.00000000,0.0000000,0.00000000
1,0.000000000,0.00000000,0.4329299,0.00000000
2,0.000000000,0.07937655,0.5123064,0.00000000
3,-0.007303067,0.11630382,0.5497721,0.00000000
4,0.037686411,0.22368302,0.6460540,0.09631987


,x1_f46,x2_f46,x3_f46,x4_f46
0,0.000000000,0.00000000,0.0000000,0.00000000
1,0.000000000,0.00000000,0.4329299,0.00000000
2,0.000000000,0.07937655,0.5123064,0.00000000
3,-0.007303067,0.11630382,0.5497721,0.00000000
4,0.000000000,0.13373451,0.5654013,0.01563544
5,0.000000000,0.17828899,0.6066248,0.05687138
6,0.037686411,0.22368302,0.6460540,0.09631987


,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,⋯,s62,s63,s64,s65,s66,s67,s68,s69,s70,s71
x1_f46,0,0.0000000,0.00000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000,⋯,0.02827311,0.02904884,0.02979361,0.03048638,0.03092923,0.03163398,0.03203542,0.03246456,0.03288872,0.03329325
x2_f46,0,0.0000000,0.00000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000,⋯,0.21236521,0.21330012,0.21419622,0.21502920,0.21557767,0.21641122,0.21690640,0.21742451,0.21793464,0.21842028
x3_f46,0,0.0480221,0.09177804,0.1316468,0.1679738,0.2010735,0.2312328,0.2587128,0.2837516,0.306566,⋯,0.63630563,0.63712015,0.63789462,0.63861232,0.63910888,0.63981211,0.64025330,0.64070703,0.64114843,0.64156588
x4_f46,0,0.0000000,0.00000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000,0.000000,⋯,0.08659350,0.08740921,0.08818279,0.08889895,0.08940471,0.09009909,0.09054618,0.09100132,0.09144226,0.09185835


## 8. Prepare data for Figures 7–9 and save CSV

In [9]:
# Figures 7, 8, 9 data --------------------------------------------------------

x1_f79  <- standardize_for_path(c(10,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0))
x2_f79  <- standardize_for_path(c(0,20,0,0,0,0,0,0,0,0,0,0,0,0,0,0))
x3_f79  <- standardize_for_path(c(0,0,30,0,0,0,0,0,0,0,0,0,0,0,0,0))
x4_f79  <- standardize_for_path(c(0,0,0,40,0,0,0,0,0,0,0,0,0,0,0,0))
x5_f79  <- standardize_for_path(c(0,0,0,0,50,0,0,0,0,0,0,0,0,0,0,0))
x6_f79  <- standardize_for_path(c(0,0,0,0,0,60,0,0,0,0,0,0,0,0,0,0))
x7_f79  <- standardize_for_path(c(0,0,0,0,0,0,70,0,0,0,0,0,0,0,0,0))
x8_f79  <- standardize_for_path(c(0,0,0,0,0,0,0,80,0,0,0,0,0,0,0,0))
x9_f79  <- standardize_for_path(c(0,0,0,0,0,0,0,0,9,0,0,0,0,0,0,0))
x10_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,10,0,0,0,0,0,0))
x11_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0))
x12_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,12,0,0,0,0))
x13_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,13,0,0,0))
x14_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,0,14,0,0))
x15_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,0,0,15,0))
x16_f79 <- standardize_for_path(c(0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,16))

X_f79 <- cbind(
  x1_f79, x2_f79, x3_f79, x4_f79, x5_f79, x6_f79, x7_f79, x8_f79,
  x9_f79, x10_f79, x11_f79, x12_f79, x13_f79, x14_f79, x15_f79, x16_f79
)

y_f79 <- c(-1,10,-5,-10,1,8,3,4,2,1,9,7,5,3,2,-1)
y_f79 <- y_f79 - mean(y_f79)

figure79_data <- data.frame(
  obs = 1:16,
  x1 = x1_f79, x2 = x2_f79, x3 = x3_f79, x4 = x4_f79,
  x5 = x5_f79, x6 = x6_f79, x7 = x7_f79, x8 = x8_f79,
  x9 = x9_f79, x10 = x10_f79, x11 = x11_f79, x12 = x12_f79,
  x13 = x13_f79, x14 = x14_f79, x15 = x15_f79, x16 = x16_f79,
  y = y_f79
)

write.csv(
  figure79_data,
  file = file.path("data", "figure7_9_data.csv"),
  row.names = FALSE
)

figure79_data

obs,x1,x2,x3,x4,x5,x6,x7,x8,x9,x10,x11,x12,x13,x14,x15,x16,y
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-3.375
2,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,7.625
3,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-7.375
4,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-12.375
5,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.375
6,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,5.625
7,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,0.625
8,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,1.625
9,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,15.491933,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-1.032796,-0.375


## 9. Generate and save Figures 7–9 as PDF

In [10]:
# Figures 7, 8, 9 ------------------------------------------------------------

c_lasso_79 <- lars(X_f79, y_f79, type = "lasso")
a_lar_79 <- lars(X_f79, y_f79, type = "lar")
b_glmnet_79 <- glmnet(X_f79, y_f79, alpha = 1, family = "gaussian")

save_plot_pdf("figure7_lasso_path_singletons.pdf", plot(c_lasso_79))
save_plot_pdf("figure8_lar_path_singletons.pdf", plot(a_lar_79))
save_plot_pdf("figure9_glmnet_path_singletons.pdf", plot(b_glmnet_79))

lar_beta_79 <- lars(X_f79, y_f79, type = "lar")$beta
lasso_beta_79 <- lars(X_f79, y_f79, type = "lasso")$beta
glmnet_beta_79 <- as.matrix(glmnet(X_f79, y_f79, alpha = 1, family = "gaussian")$beta)

lar_beta_79
lasso_beta_79
glmnet_beta_79

write.csv(lar_beta_79, file = file.path("outputs", "figure7_9_lar_beta.csv"), row.names = FALSE)
write.csv(lasso_beta_79, file = file.path("outputs", "figure7_9_lasso_beta.csv"), row.names = FALSE)
write.csv(glmnet_beta_79, file = file.path("outputs", "figure7_9_glmnet_beta.csv"), row.names = TRUE)

,x1_f79,x2_f79,x3_f79,x4_f79,x5_f79,x6_f79,x7_f79,x8_f79,x9_f79,x10_f79,x11_f79,x12_f79,x13_f79,x14_f79,x15_f79,x16_f79
0,0.00000000,0.00000000,0.00000000,0.0000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
1,0.00000000,0.00000000,0.00000000,-0.3025768,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
2,0.00000000,0.00000000,-0.03025768,-0.3328345,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
3,0.00000000,0.06051536,-0.10085894,-0.4034358,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
4,0.00000000,0.12103073,-0.16137431,-0.4639511,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.06051536,0.00000000,0.00000000,0.00000000,0,0.00000000
5,0.00000000,0.18154609,-0.21180378,-0.5143806,0.00000000,0.06051536,0.00000000,0.00000000,0.000000e+00,0.00000000,0.12103073,0.00000000,0.00000000,0.00000000,0,0.00000000
6,0.00000000,0.22693262,-0.24206146,-0.5446383,0.00000000,0.10590189,0.00000000,0.00000000,0.000000e+00,0.00000000,0.16641725,0.04538652,0.00000000,0.00000000,0,0.00000000
7,-0.07564421,0.30257682,-0.31770567,-0.6202825,0.00000000,0.18154609,0.00000000,0.00000000,0.000000e+00,0.00000000,0.24206146,0.12103073,0.00000000,0.00000000,0,-0.07564421
8,-0.12103073,0.36309219,-0.36309219,-0.6656690,0.00000000,0.24206146,0.00000000,0.00000000,0.000000e+00,0.00000000,0.30257682,0.18154609,0.06051536,0.00000000,0,-0.12103073
9,-0.18154609,0.42360755,-0.42360755,-0.7261844,-0.06051536,0.30257682,0.00000000,0.06051536,0.000000e+00,-0.06051536,0.36309219,0.24206146,0.12103073,0.00000000,0,-0.18154609


,x1_f79,x2_f79,x3_f79,x4_f79,x5_f79,x6_f79,x7_f79,x8_f79,x9_f79,x10_f79,x11_f79,x12_f79,x13_f79,x14_f79,x15_f79,x16_f79
0,0.00000000,0.00000000,0.00000000,0.0000000,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
1,0.00000000,0.00000000,0.00000000,-0.3025768,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
2,0.00000000,0.00000000,-0.03025768,-0.3328345,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
3,0.00000000,0.06051536,-0.10085894,-0.4034358,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,0,0.00000000
4,0.00000000,0.12103073,-0.16137431,-0.4639511,0.00000000,0.00000000,0.00000000,0.00000000,0.000000e+00,0.00000000,0.06051536,0.00000000,0.00000000,0.00000000,0,0.00000000
5,0.00000000,0.18154609,-0.21180378,-0.5143806,0.00000000,0.06051536,0.00000000,0.00000000,0.000000e+00,0.00000000,0.12103073,0.00000000,0.00000000,0.00000000,0,0.00000000
6,0.00000000,0.22693262,-0.24206146,-0.5446383,0.00000000,0.10590189,0.00000000,0.00000000,0.000000e+00,0.00000000,0.16641725,0.04538652,0.00000000,0.00000000,0,0.00000000
7,-0.07564421,0.30257682,-0.31770567,-0.6202825,0.00000000,0.18154609,0.00000000,0.00000000,0.000000e+00,0.00000000,0.24206146,0.12103073,0.00000000,0.00000000,0,-0.07564421
8,-0.12103073,0.36309219,-0.36309219,-0.6656690,0.00000000,0.24206146,0.00000000,0.00000000,0.000000e+00,0.00000000,0.30257682,0.18154609,0.06051536,0.00000000,0,-0.12103073
9,-0.18154609,0.42360755,-0.42360755,-0.7261844,-0.06051536,0.30257682,0.00000000,0.06051536,0.000000e+00,-0.06051536,0.36309219,0.24206146,0.12103073,0.00000000,0,-0.18154609


,s0,s1,s2,s3,s4,s5,s6,s7,s8,s9,⋯,s38,s39,s40,s41,s42,s43,s44,s45,s46,s47
x1_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.189523031,-0.191365207,-0.19304373,-0.19457313,-0.19596667,-0.19723641,-0.19839335,-0.19944751,-0.20040802,-0.20128321
x2_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.009416637,0.04455803,0.07791482,0.10952073,⋯,0.432486580,0.434522985,0.43637848,0.43806914,0.43960961,0.44101322,0.44229215,0.44345745,0.44451924,0.44548669
x3_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,-0.041243202,-0.08224161,-0.11826045,-0.14986711,⋯,-0.431583586,-0.433425842,-0.43510444,-0.43663391,-0.43802751,-0.43929730,-0.44045429,-0.44150850,-0.44246905,-0.44334427
x4_f79,0,-0.07096344,-0.1356227,-0.1945378,-0.248219,-0.2971314,-0.343820247,-0.38481860,-0.42083652,-0.45244290,⋯,-0.734158973,-0.736001356,-0.73768007,-0.73920965,-0.74060334,-0.74187323,-0.74303030,-0.74408458,-0.74504520,-0.74592048
x5_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.068490109,-0.070332479,-0.07201118,-0.07354075,-0.07493443,-0.07620431,-0.07736137,-0.07841564,-0.07937625,-0.08025153
x6_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,0.311456567,0.313492909,0.31534835,0.31703895,0.31857937,0.31998294,0.32126183,0.32242710,0.32348885,0.32445627
x7_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,0.008879842,0.010916175,0.01277161,0.01446221,0.01600262,0.01740618,0.01868506,0.01985032,0.02091207,0.02187949
x8_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,0.069395285,0.071431611,0.07328704,0.07497763,0.07651804,0.07792160,0.07920047,0.08036573,0.08142747,0.08239489
x9_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.007975203,-0.009817532,-0.01149619,-0.01302573,-0.01441938,-0.01568923,-0.01684627,-0.01790051,-0.01886111,-0.01973636
x10_f79,0,0.00000000,0.0000000,0.0000000,0.000000,0.0000000,0.000000000,0.00000000,0.00000000,0.00000000,⋯,-0.068490790,-0.070333099,-0.07201174,-0.07354126,-0.07493490,-0.07620473,-0.07736176,-0.07841599,-0.07937658,-0.08025182


## 10. Real-life UCI clustering datasets: setup

This section adds five real-life UCI datasets for unsupervised model selection in clustering.
It compares the **original standardized data** with a **sparsified version** of the same data.

Expected dataset stems:
- `breast_tissue`
- `dry_bean`
- `hcv`
- `raisin`
- `wholesale_customers`

**Important:** this revised version assumes that the **class/label column is the last column in each dataset**.
The notebook will therefore use the **last column as the label by default**, and it will use name-based matching only as a fallback.

The notebook will automatically look for `.csv`, `.xlsx`, or `.xls` files in `data/` first and then in the current working directory.



In [23]:
# Real-life UCI clustering: setup --------------------------------------------

options(timeout = 600)

install_if_needed <- function(pkgs, repos = "https://cloud.r-project.org") {
  installed <- rownames(installed.packages())
  to_install <- pkgs[!pkgs %in% installed]
  if (length(to_install) > 0) {
    install.packages(to_install, repos = repos)
  }
}

install_if_needed(c("readxl", "cluster", "mclust", "clue"))

library(readxl)
library(cluster)
library(mclust)
library(clue)

sanitize_names <- function(x) {
  x <- tolower(trimws(x))
  x <- gsub("[^a-z0-9]+", "_", x)
  x <- gsub("^_+|_+$", "", x)
  x
}

find_data_file <- function(stem, search_dirs = c("data", "."), exts = c("csv", "xlsx", "xls")) {
  for (d in search_dirs) {
    if (!dir.exists(d) && d != ".") next

    for (ext in exts) {
      candidate <- file.path(d, paste0(stem, ".", ext))
      if (file.exists(candidate)) return(candidate)
    }

    pattern <- paste0("^", stem, "\\.(", paste(exts, collapse = "|"), ")$")
    matches <- list.files(d, pattern = pattern, ignore.case = TRUE, full.names = TRUE)
    if (length(matches) > 0) return(matches[1])
  }
  NA_character_
}

read_tabular_file <- function(path) {
  if (is.na(path)) stop("File not found.")
  ext <- tolower(tools::file_ext(path))

  if (ext %in% c("xlsx", "xls")) {
    as.data.frame(readxl::read_excel(path))
  } else {
    read.csv(path, check.names = FALSE, stringsAsFactors = FALSE)
  }
}

most_frequent_value <- function(x) {
  x <- x[!is.na(x)]
  if (length(x) == 0) return(NA)
  ux <- unique(x)
  ux[which.max(tabulate(match(x, ux)))]
}

impute_missing_values <- function(df) {
  for (j in seq_along(df)) {
    if (is.numeric(df[[j]])) {
      med <- stats::median(df[[j]], na.rm = TRUE)
      if (is.na(med)) med <- 0
      df[[j]][is.na(df[[j]])] <- med
    } else {
      mode_val <- most_frequent_value(df[[j]])
      df[[j]][is.na(df[[j]])] <- mode_val
      df[[j]] <- as.factor(df[[j]])
    }
  }
  df
}

SPARSE_ORTHO_BINS <- 4

# Sparse orthogonalization-style transform: each standardized variable is
# partitioned into non-overlapping quantile blocks, and each block becomes
# a separate sparse component. This is closer to the sparse expanded modeling
# used in the regression phase than simple hard-thresholding.
sparse_orthogonalize_matrix <- function(X_scaled, n_blocks = SPARSE_ORTHO_BINS, rescale_output = TRUE) {
  X_scaled <- as.matrix(X_scaled)
  out_list <- list()
  out_names <- character(0)

  for (j in seq_len(ncol(X_scaled))) {
    x <- X_scaled[, j]
    probs <- seq(0, 1, length.out = n_blocks + 1)
    brks <- unique(as.numeric(stats::quantile(x, probs = probs, na.rm = TRUE, type = 8)))

    if (length(brks) < 2) next

    bin_id <- cut(x, breaks = brks, include.lowest = TRUE, labels = FALSE)
    if (all(is.na(bin_id))) next

    nb <- max(bin_id, na.rm = TRUE)
    for (b in seq_len(nb)) {
      comp <- numeric(length(x))
      idx <- which(bin_id == b)
      if (length(idx) == 0) next
      comp[idx] <- x[idx]
      out_list[[length(out_list) + 1]] <- comp
      out_names <- c(out_names, paste0(colnames(X_scaled)[j], "_b", b))
    }
  }

  if (length(out_list) == 0) return(X_scaled)

  X_sparse <- do.call(cbind, out_list)
  colnames(X_sparse) <- out_names

  keep_cols <- apply(X_sparse, 2, function(z) stats::sd(z, na.rm = TRUE) > 0)
  X_sparse <- X_sparse[, keep_cols, drop = FALSE]

  if (rescale_output && ncol(X_sparse) > 0) {
    X_sparse <- scale(X_sparse)
    X_sparse[is.na(X_sparse)] <- 0
  }

  X_sparse
}

matrix_sparsity_ratio <- function(X) {
  mean(abs(X) < .Machine$double.eps)
}

stratified_sample_indices <- function(labels, max_n) {
  n <- length(labels)
  if (n <= max_n) return(seq_len(n))

  labels <- as.character(labels)
  labels[is.na(labels) | labels == ""] <- "__missing__"
  labels <- factor(labels)

  split_idx <- split(seq_len(n), labels)
  grp_sizes <- sapply(split_idx, length)

  props <- grp_sizes / sum(grp_sizes)
  alloc <- round(props * max_n)
  alloc <- pmax(1, alloc)
  alloc <- pmin(alloc, grp_sizes)

  names(alloc) <- names(grp_sizes)

  while (sum(alloc) > max_n) {
    reducible <- which(alloc > 1)
    if (length(reducible) == 0) break
    ix <- reducible[which.max(alloc[reducible])]
    alloc[ix] <- alloc[ix] - 1
  }

  while (sum(alloc) < max_n) {
    expandable <- which(alloc < grp_sizes)
    if (length(expandable) == 0) break
    need <- grp_sizes[expandable] - alloc[expandable]
    ix <- expandable[which.max(need)]
    alloc[ix] <- alloc[ix] + 1
  }

  out <- integer(0)
  for (nm in names(split_idx)) {
    idx <- split_idx[[nm]]
    take <- alloc[[nm]]
    if (is.na(take) || take <= 0 || length(idx) == 0) next
    out <- c(out, sample(idx, min(length(idx), take)))
  }

  sort(unique(out))
}

normalized_mutual_information <- function(labels_pred, labels_true) {
  tab <- table(as.factor(labels_pred), as.factor(labels_true))
  nij <- tab / sum(tab)
  pi <- rowSums(nij)
  pj <- colSums(nij)

  mi <- 0
  for (i in seq_len(nrow(nij))) {
    for (j in seq_len(ncol(nij))) {
      if (nij[i, j] > 0 && pi[i] > 0 && pj[j] > 0) {
        mi <- mi + nij[i, j] * log(nij[i, j] / (pi[i] * pj[j]))
      }
    }
  }

  hi <- -sum(pi[pi > 0] * log(pi[pi > 0]))
  hj <- -sum(pj[pj > 0] * log(pj[pj > 0]))
  if (hi <= 0 || hj <= 0) return(NA_real_)

  mi / sqrt(hi * hj)
}

clustering_purity <- function(labels_pred, labels_true) {
  if (any(is.na(labels_true))) {
    stop("True class labels contain NA values.")
  }
  if (any(is.na(labels_pred))) {
    return(NA_real_)
  }
  if (length(labels_pred) != length(labels_true)) {
    return(NA_real_)
  }

  tab <- table(as.factor(labels_pred), as.factor(labels_true))
  if (length(tab) == 0 || nrow(tab) == 0 || ncol(tab) == 0) return(NA_real_)

  sum(apply(tab, 1, max)) / sum(tab)
}

clustering_accuracy <- function(labels_pred, labels_true) {
  if (any(is.na(labels_true))) {
    stop("True class labels contain NA values.")
  }
  if (any(is.na(labels_pred))) {
    return(NA_real_)
  }
  if (length(labels_pred) != length(labels_true)) {
    return(NA_real_)
  }

  tab <- table(as.factor(labels_pred), as.factor(labels_true))
  if (length(tab) == 0 || nrow(tab) == 0 || ncol(tab) == 0) return(NA_real_)

  nr <- nrow(tab)
  nc <- ncol(tab)
  nmax <- max(nr, nc)

  score_mat <- matrix(0, nrow = nmax, ncol = nmax)
  score_mat[seq_len(nr), seq_len(nc)] <- tab

  assignment <- clue::solve_LSAP(score_mat, maximum = TRUE)
  matched <- sum(score_mat[cbind(seq_len(nmax), assignment)])
  matched / sum(tab)
}
davies_bouldin_index <- function(X, labels) {
  labels <- as.factor(labels)
  k <- nlevels(labels)
  if (k < 2) return(NA_real_)

  centers <- sapply(levels(labels), function(cl) {
    colMeans(X[labels == cl, , drop = FALSE])
  })
  if (is.vector(centers)) centers <- matrix(centers, ncol = k)

  S <- sapply(levels(labels), function(cl) {
    Xi <- X[labels == cl, , drop = FALSE]
    ci <- colMeans(Xi)
    mean(sqrt(rowSums((sweep(Xi, 2, ci))^2)))
  })

  M <- as.matrix(dist(t(centers)))
  R <- matrix(NA_real_, nrow = k, ncol = k)

  for (i in seq_len(k)) {
    for (j in seq_len(k)) {
      if (i != j) {
        if (M[i, j] == 0) {
          R[i, j] <- NA_real_
        } else {
          R[i, j] <- (S[i] + S[j]) / M[i, j]
        }
      }
    }
  }

  mean(apply(R, 1, max, na.rm = TRUE), na.rm = TRUE)
}

calinski_harabasz_index <- function(X, labels) {
  labels <- as.factor(labels)
  n <- nrow(X)
  k <- nlevels(labels)
  if (k < 2 || k >= n) return(NA_real_)

  overall_mean <- colMeans(X)
  B <- 0
  W <- 0

  for (cl in levels(labels)) {
    Xi <- X[labels == cl, , drop = FALSE]
    ni <- nrow(Xi)
    if (ni <= 0) next
    ci <- colMeans(Xi)
    B <- B + ni * sum((ci - overall_mean)^2)
    W <- W + sum(rowSums((sweep(Xi, 2, ci))^2))
  }

  if (W == 0) return(NA_real_)
  (B / (k - 1)) / (W / (n - k))
}

prepare_dataset_for_clustering <- function(df, dataset_key, label_candidates = NULL,
                                           drop_candidates = NULL, prefer_last_column = TRUE) {
  names(df) <- sanitize_names(names(df))

  if (is.null(label_candidates)) {
    label_candidates <- c("class", "label", "category", "target", "channel")
  }
  if (is.null(drop_candidates)) {
    drop_candidates <- character(0)
  }

  last_col <- names(df)[ncol(df)]

  if (prefer_last_column) {
    label_col <- last_col
  } else {
    matches <- label_candidates[label_candidates %in% names(df)]
    label_col <- if (length(matches) > 0) matches[1] else last_col
  }

  candidate_drop <- unique(c(drop_candidates, "id", "case", "case_number", "sample_id", "unnamed_0"))
  drop_cols <- candidate_drop[candidate_drop %in% names(df)]

  y <- as.factor(df[[label_col]])
  X_df <- df[, setdiff(names(df), c(label_col, drop_cols)), drop = FALSE]
  X_df <- impute_missing_values(X_df)

  mm <- model.matrix(~ . - 1, data = X_df)
  mm <- as.matrix(mm)

  keep_cols <- apply(mm, 2, function(z) stats::sd(z, na.rm = TRUE) > 0)
  mm <- mm[, keep_cols, drop = FALSE]

  X_scaled <- scale(mm)
  X_sparse <- sparse_orthogonalize_matrix(X_scaled, n_blocks = SPARSE_ORTHO_BINS)

  list(
    labels = y,
    X_original = X_scaled,
    X_sparsified = X_sparse,
    n = nrow(mm),
    p = ncol(mm),
    p_sparse = ncol(X_sparse),
    true_k = nlevels(y),
    label_col = label_col,
    label_position = ncol(df),
    drop_cols = paste(drop_cols, collapse = ";"),
    original_sparsity = matrix_sparsity_ratio(X_scaled),
    sparsified_sparsity = matrix_sparsity_ratio(X_sparse)
  )
}

run_clustering_method <- function(X, method_name, k) {
  if (method_name == "kmeans") {
    fit <- kmeans(X, centers = k, nstart = 30, iter.max = 100)
    return(list(cluster = fit$cluster))
  } else if (method_name == "hclust") {
    hc <- hclust(dist(X), method = "ward.D2")
    return(list(cluster = cutree(hc, k = k)))
  } else if (method_name == "gmm") {
    fit <- tryCatch(
      mclust::Mclust(
        X,
        G = k,
        verbose = FALSE,
        control = mclust::emControl(itmax = 200)
      ),
      error = function(e) NULL,
      warning = function(w) {
        invokeRestart("muffleWarning")
      }
    )

    if (is.null(fit) || is.null(fit$classification)) return(NULL)
    return(list(cluster = fit$classification))
  } else {
    stop("Unsupported clustering method.")
  }
}

compute_metrics_for_fit <- function(X, labels_pred, labels_true = NULL, silhouette_sample_n = 1000) {
  if (length(labels_pred) != nrow(X)) {
    return(list(
      silhouette = NA_real_,
      davies_bouldin = NA_real_,
      calinski_harabasz = NA_real_,
      ari = NA_real_,
      nmi = NA_real_,
      accuracy = NA_real_,
      purity = NA_real_
    ))
  }

  if (any(is.na(labels_pred))) {
    return(list(
      silhouette = NA_real_,
      davies_bouldin = NA_real_,
      calinski_harabasz = NA_real_,
      ari = NA_real_,
      nmi = NA_real_,
      accuracy = NA_real_,
      purity = NA_real_
    ))
  }

  sil <- NA_real_
  if (nrow(X) > 1 && length(unique(labels_pred)) > 1) {
    idx <- seq_len(nrow(X))
    if (nrow(X) > silhouette_sample_n) {
      idx <- sample(idx, silhouette_sample_n)
    }

    X_sub <- X[idx, , drop = FALSE]
    labels_sub <- labels_pred[idx]

    sil <- tryCatch({
      d_sub <- dist(X_sub)
      sil_obj <- cluster::silhouette(labels_sub, d_sub)
      sil_mat <- unclass(sil_obj)

      if (is.null(dim(sil_mat)) || ncol(sil_mat) < 3) {
        NA_real_
      } else {
        mean(sil_mat[, 3], na.rm = TRUE)
      }
    }, error = function(e) {
      NA_real_
    })
  }

  db <- if (nrow(X) > 1 && length(unique(labels_pred)) > 1) {
    tryCatch(davies_bouldin_index(X, labels_pred), error = function(e) NA_real_)
  } else {
    NA_real_
  }

  ch <- if (nrow(X) > 1 && length(unique(labels_pred)) > 1) {
    tryCatch(calinski_harabasz_index(X, labels_pred), error = function(e) NA_real_)
  } else {
    NA_real_
  }

  ari <- NA_real_
  nmi <- NA_real_
  acc <- NA_real_
  purity <- NA_real_

  if (!is.null(labels_true)) {
    if (any(is.na(labels_true))) {
      stop("True class labels contain NA values.")
    }

    if (length(labels_true) == length(labels_pred)) {
      ari <- tryCatch(
        mclust::adjustedRandIndex(labels_pred, labels_true),
        error = function(e) NA_real_
      )
      nmi <- tryCatch(
        normalized_mutual_information(labels_pred, labels_true),
        error = function(e) NA_real_
      )
      acc <- tryCatch(
        clustering_accuracy(labels_pred, labels_true),
        error = function(e) NA_real_
      )
      purity <- tryCatch(
        clustering_purity(labels_pred, labels_true),
        error = function(e) NA_real_
      )
    }
  }

  list(
    silhouette = sil,
    davies_bouldin = db,
    calinski_harabasz = ch,
    ari = ari,
    nmi = nmi,
    accuracy = acc,
    purity = purity
  )
}

evaluate_clustering_grid <- function(X, labels = NULL, dataset_name, representation_name,
                                     k_values, methods = c("kmeans"),
                                     method_limits = c(kmeans = Inf)) {
  results <- data.frame()
  assignments <- list()

  for (method_name in methods) {
    limit_n <- method_limits[method_name]
    idx_used <- seq_len(nrow(X))

    if (is.finite(limit_n) && nrow(X) > limit_n) {
      if (is.null(labels)) {
        idx_used <- sort(sample(seq_len(nrow(X)), limit_n))
      } else {
        idx_used <- stratified_sample_indices(labels, limit_n)
      }
    }

    X_use <- X[idx_used, , drop = FALSE]
    labels_use <- if (!is.null(labels)) labels[idx_used] else NULL

    for (k in k_values) {
      cat(sprintf(
        "Running dataset=%s | representation=%s | method=%s | k=%d | n=%d | p=%d\n",
        dataset_name, representation_name, method_name, k, nrow(X_use), ncol(X_use)
      ))

      timed <- system.time({
        fit <- tryCatch(
          run_clustering_method(X_use, method_name, k),
          error = function(e) {
            cat(sprintf(
              "  -> ERROR: dataset=%s | representation=%s | method=%s | k=%d | msg=%s\n",
              dataset_name, representation_name, method_name, k, e$message
            ))
            NULL
          }
        )
      })

      if (is.null(fit) || is.null(fit$cluster)) {
        cat(sprintf(
          "  -> SKIPPED: dataset=%s | representation=%s | method=%s | k=%d | invalid fit\n",
          dataset_name, representation_name, method_name, k
        ))
        next
      }

      labels_pred <- fit$cluster
      if (length(labels_pred) != nrow(X_use)) {
        cat(sprintf(
          "  -> SKIPPED: dataset=%s | representation=%s | method=%s | k=%d | cluster length=%d, expected=%d\n",
          dataset_name, representation_name, method_name, k, length(labels_pred), nrow(X_use)
        ))
        next
      }

      metric <- compute_metrics_for_fit(X_use, labels_pred, labels_use)

      results <- rbind(
        results,
        data.frame(
          dataset = dataset_name,
          representation = representation_name,
          method = method_name,
          k = k,
          n_used = nrow(X_use),
          silhouette = metric$silhouette,
          davies_bouldin = metric$davies_bouldin,
          calinski_harabasz = metric$calinski_harabasz,
          adjusted_rand_index = metric$ari,
          normalized_mutual_information = metric$nmi,
          clustering_accuracy = metric$accuracy,
          purity = metric$purity,
          runtime_sec = unname(timed["elapsed"]),
          stringsAsFactors = FALSE
        )
      )

      assignments[[paste0(method_name, "_k", k)]] <- data.frame(
        obs_index = idx_used,
        predicted_cluster = labels_pred,
        stringsAsFactors = FALSE
      )
    }
  }

  list(results = results, assignments = assignments)
}

plot_metric_pairs_fixed_xlim <- function(summary_df, value_col, main_title, xlab, xlim_fixed) {
  methods <- unique(summary_df$method)
  op <- par(mfrow = c(length(methods), 1), mar = c(5, 20, 5.5, 2), xpd = NA)
  on.exit(par(op), add = TRUE)

  for (m in methods) {
    sub <- summary_df[summary_df$method == m, ]
    ds <- unique(sub$dataset)
    ds_display <- gsub("_", " ", ds)

    orig_vals <- sapply(ds, function(d) sub[sub$dataset == d & sub$representation == "original", value_col][1])
    spar_vals <- sapply(ds, function(d) sub[sub$dataset == d & sub$representation == "sparsified", value_col][1])

    y_pos <- seq_along(ds)

    plot(orig_vals, y_pos,
         pch = 16, col = "black", yaxt = "n",
         xlab = xlab, ylab = "",
         main = paste(main_title, "-", m),
         xlim = xlim_fixed)

    axis(2, at = y_pos, labels = ds_display, las = 2)
    points(spar_vals, y_pos, pch = 17, col = "red")

    for (i in seq_along(ds)) {
      segments(orig_vals[i], y_pos[i], spar_vals[i], y_pos[i], col = "gray60", lty = 2)
    }

    legend(
      "topright",
      inset = c(0.00, -0.12),
      legend = c("Original", "Sparsified"),
      pch = c(16, 17),
      col = c("black", "red"),
      horiz = TRUE,
      bty = "n",
      pt.cex = 1.1,
      cex = 1
    )
  }
}
plot_metric_pairs <- function(summary_df, value_col, main_title, xlab, higher_is_better = TRUE) {
  methods <- unique(summary_df$method)
  op <- par(mfrow = c(length(methods), 1), mar = c(5, 20, 5.5, 2), xpd = NA)
  on.exit(par(op), add = TRUE)

  for (m in methods) {
    sub <- summary_df[summary_df$method == m, ]
    ds <- unique(sub$dataset)
    ds_display <- gsub("_", " ", ds)

    orig_vals <- sapply(ds, function(d) sub[sub$dataset == d & sub$representation == "original", value_col][1])
    spar_vals <- sapply(ds, function(d) sub[sub$dataset == d & sub$representation == "sparsified", value_col][1])

    y_pos <- seq_along(ds)
    xlim <- range(c(orig_vals, spar_vals), na.rm = TRUE)
    if (!all(is.finite(xlim))) xlim <- c(0, 1)

    plot(orig_vals, y_pos,
         pch = 16, col = "black", yaxt = "n",
         xlab = xlab, ylab = "",
         main = paste(main_title, "-", m),
         xlim = xlim)

    axis(2, at = y_pos, labels = ds_display, las = 2)
    points(spar_vals, y_pos, pch = 17, col = "red")

    for (i in seq_along(ds)) {
      segments(orig_vals[i], y_pos[i], spar_vals[i], y_pos[i], col = "gray60", lty = 2)
    }

    legend(
      "topright",
      inset = c(0.00, -0.12),
      legend = c("Original", "Sparsified"),
      pch = c(16, 17),
      col = c("black", "red"),
      horiz = TRUE,
      bty = "n",
      pt.cex = 1.1,
      cex = 1
    )
  }
}

plot_sparsity_summary <- function(df) {
  ord <- order(df$sparsified_sparsity - df$original_sparsity, decreasing = TRUE)
  df <- df[ord, ]

  barplot(
    rbind(df$original_sparsity, df$sparsified_sparsity),
    beside = TRUE,
    names.arg = df$dataset,
    las = 2,
    ylim = c(0, 1),
    col = c("gray70", "gray30"),
    main = "Sparsity before and after sparsification",
    ylab = "Proportion of zero entries"
  )

  legend("topleft", legend = c("Original", "Sparsified"),
         fill = c("gray70", "gray30"), bty = "n")
}

find_data_file_multi <- function(stems, search_dirs = c("data", "."), exts = c("csv", "xlsx", "xls")) {
  for (stem in stems) {
    f <- find_data_file(stem, search_dirs = search_dirs, exts = exts)
    if (!is.na(f)) return(f)
  }
  NA_character_
}


## 11. Load and preprocess the 14 UCI datasets

This version uses the **last column as the class/label column** for each uploaded UCI dataset.
The class labels are used only for external evaluation, not for clustering itself.



In [12]:
# Load and preprocess the 12 UCI datasets (sparse orthogonalized version) -----

uci_specs <- list(
  breast_tissue = list(
    aliases = c("breast_tissue"),
    label_candidates = c("class", "label"),
    drop_candidates = c("case", "case_number")
  ),
  hcv = list(
    aliases = c("hcv"),
    label_candidates = c("class", "category", "label"),
    drop_candidates = c("unnamed_0")
  ),
  wholesale_customers = list(
    aliases = c("wholesale_customers"),
    label_candidates = c("class", "label", "channel"),
    drop_candidates = c("region", "customer_id")
  ),
  pen_recognition_handwritten_digits = list(
    aliases = c("pen_recognition_handwritten_digits"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  libras_movement = list(
    aliases = c("libras_movement"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  vehicle_silhouettes = list(
    aliases = c("vehicle_silhouettes", "vehicle_silhouette"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  statlog_image_segmentation = list(
    aliases = c("statlog_image_segmentation", "image_segmentation", "statlog_segment"),
    label_candidates = c("class", "label"),
    drop_candidates = c("region_pixel_count")
  ),
  vertebral_column_3classes = list(
    aliases = c("vertebral_column_3classes", "vertebral_column_3_classes", "vertebral_3classes"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  vertebral_column_2classes = list(
    aliases = c("vertebral_column_2classes", "vertebral_column_2_classes", "vertebral_2classes"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  user_knowledge_modeling = list(
    aliases = c("user_knowledge_modeling", "user_knowledge_modelling"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  banknote_authentication = list(
    aliases = c("banknote_authentication", "banknote_authentication_dataset", "banknote_auth"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  ),
  wine = list(
    aliases = c("wine"),
    label_candidates = c("class", "label"),
    drop_candidates = character(0)
  )
)

uci_prepared <- list()
uci_overview <- data.frame()

for (nm in names(uci_specs)) {
  spec <- uci_specs[[nm]]

  path <- find_data_file_multi(spec$aliases)
  if (is.na(path)) {
    warning(sprintf("Dataset file not found for %s", nm))
    next
  }

  raw_df <- read_tabular_file(path)

  prep <- prepare_dataset_for_clustering(
    raw_df,
    dataset_key = nm,
    label_candidates = spec$label_candidates,
    drop_candidates = spec$drop_candidates,
    prefer_last_column = TRUE
  )

  if (any(is.na(prep$labels))) {
    stop(sprintf("Dataset %s has NA in the class column.", nm))
  }

  cat(sprintf("Prepared dataset: %s | n = %d | p = %d | true_k = %d\n",
              nm, prep$n, prep$p, prep$true_k))

  uci_prepared[[nm]] <- prep

  uci_overview <- rbind(
    uci_overview,
    data.frame(
      dataset = nm,
      file_path = path,
      n = prep$n,
      p_after_encoding = prep$p,
      p_sparse_orthogonalized = prep$p_sparse,
      label_col = prep$label_col,
      label_position = prep$label_position,
      dropped_cols = prep$drop_cols,
      true_k = prep$true_k,
      original_sparsity = round(prep$original_sparsity, 4),
      sparsified_sparsity = round(prep$sparsified_sparsity, 4),
      stringsAsFactors = FALSE
    )
  )
}

write.csv(uci_overview, "outputs/uci_dataset_overview.csv", row.names = FALSE)

cat("Datasets loaded:", paste(names(uci_prepared), collapse = ", "), "\n")
cat("Label handling: the last column is used as the class column for each dataset.\n")
cat(sprintf("Sparse orthogonalization blocks per feature: %d\n", SPARSE_ORTHO_BINS))

uci_overview

Prepared dataset: breast_tissue | n = 106 | p = 9 | true_k = 6
Prepared dataset: hcv | n = 615 | p = 13 | true_k = 5
Prepared dataset: wholesale_customers | n = 440 | p = 7 | true_k = 3
Prepared dataset: pen_recognition_handwritten_digits | n = 10992 | p = 16 | true_k = 10
Prepared dataset: libras_movement | n = 360 | p = 90 | true_k = 15
Prepared dataset: vehicle_silhouettes | n = 846 | p = 18 | true_k = 4
Prepared dataset: statlog_image_segmentation | n = 2310 | p = 18 | true_k = 7
Prepared dataset: vertebral_column_3classes | n = 310 | p = 6 | true_k = 3
Prepared dataset: vertebral_column_2classes | n = 310 | p = 6 | true_k = 2
Prepared dataset: user_knowledge_modeling | n = 403 | p = 5 | true_k = 5
Prepared dataset: banknote_authentication | n = 1372 | p = 4 | true_k = 2
Prepared dataset: wine | n = 178 | p = 13 | true_k = 3
Datasets loaded: breast_tissue, hcv, wholesale_customers, pen_recognition_handwritten_digits, libras_movement, vehicle_silhouettes, statlog_image_segmentation,

dataset,file_path,n,p_after_encoding,p_sparse_orthogonalized,label_col,label_position,dropped_cols,true_k,original_sparsity,sparsified_sparsity
<chr>,<chr>,<int>,<int>,<int>,<chr>,<int>,<chr>,<int>,<dbl>,<dbl>
breast_tissue,data/breast_tissue.csv,106,9,36,class,11,case,6,0,0
hcv,data/hcv.csv,615,13,46,class,14,id,5,0,0
wholesale_customers,data/wholesale_customers.csv,440,7,25,class,8,,3,0,0
pen_recognition_handwritten_digits,data/pen_recognition_handwritten_digits.csv,10992,16,59,class,17,,10,0,0
libras_movement,data/libras_movement.csv,360,90,360,class,91,,15,0,0
vehicle_silhouettes,data/vehicle_silhouettes.csv,846,18,72,class,19,,4,0,0
statlog_image_segmentation,data/statlog_image_segmentation.csv,2310,18,66,class,20,region_pixel_count,7,0,0
vertebral_column_3classes,data/vertebral_column_3classes.csv,310,6,24,class,7,,3,0,0
vertebral_column_2classes,data/vertebral_column_2classes.csv,310,6,24,class,7,,2,0,0


In [13]:
diag_df <- data.frame()

for (nm in names(uci_prepared)) {
  prep <- uci_prepared[[nm]]

  orig_dup <- sum(duplicated(as.data.frame(prep$X_original)))
  spar_dup <- sum(duplicated(as.data.frame(prep$X_sparsified)))

  orig_zero_rows <- sum(rowSums(abs(prep$X_original)) == 0)
  spar_zero_rows <- sum(rowSums(abs(prep$X_sparsified)) == 0)

  orig_unique_rows <- nrow(unique(as.data.frame(prep$X_original)))
  spar_unique_rows <- nrow(unique(as.data.frame(prep$X_sparsified)))

  class_tab <- table(prep$labels)

  diag_df <- rbind(
    diag_df,
    data.frame(
      dataset = nm,
      n = prep$n,
      p = prep$p,
      true_k = prep$true_k,
      label_na = sum(is.na(prep$labels)),
      xorig_na = sum(is.na(prep$X_original)),
      xsparse_na = sum(is.na(prep$X_sparsified)),
      min_class_size = min(class_tab),
      max_class_size = max(class_tab),
      original_zero_rows = orig_zero_rows,
      sparsified_zero_rows = spar_zero_rows,
      original_unique_rows = orig_unique_rows,
      sparsified_unique_rows = spar_unique_rows,
      original_duplicated_rows = orig_dup,
      sparsified_duplicated_rows = spar_dup,
      original_sparsity = prep$original_sparsity,
      sparsified_sparsity = prep$sparsified_sparsity,
      stringsAsFactors = FALSE
    )
  )
}

diag_df[order(-diag_df$sparsified_zero_rows, -diag_df$sparsified_duplicated_rows), ]

,dataset,n,p,true_k,label_na,xorig_na,xsparse_na,min_class_size,max_class_size,original_zero_rows,sparsified_zero_rows,original_unique_rows,sparsified_unique_rows,original_duplicated_rows,sparsified_duplicated_rows,original_sparsity,sparsified_sparsity
,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>
7,statlog_image_segmentation,2310,18,7,0,0,0,330,330,0,0,2086,2086,224,224,0,0
5,libras_movement,360,90,15,0,0,0,24,24,0,0,330,330,30,30,0,0
11,banknote_authentication,1372,4,2,0,0,0,610,762,0,0,1348,1348,24,24,0,0
1,breast_tissue,106,9,6,0,0,0,14,22,0,0,105,105,1,1,0,0
2,hcv,615,13,5,0,0,0,7,533,0,0,615,615,0,0,0,0
3,wholesale_customers,440,7,3,0,0,0,47,316,0,0,440,440,0,0,0,0
4,pen_recognition_handwritten_digits,10992,16,10,0,0,0,1055,1144,0,0,10992,10992,0,0,0,0
6,vehicle_silhouettes,846,18,4,0,0,0,199,218,0,0,846,846,0,0,0,0
8,vertebral_column_3classes,310,6,3,0,0,0,60,150,0,0,310,310,0,0,0,0


In [14]:
problem_datasets <- c(
  "hcv",
  "statlog_image_segmentation",
  "vehicle_silhouettes",
  "wholesale_customers",
  "banknote_authentication"
)

for (nm in problem_datasets) {
  cat("\n============================================================\n")
  cat(sprintf("TRYING DATASET: %s\n", nm))
  cat("============================================================\n")

  prep <- uci_prepared[[nm]]
  k_values <- 2:min(10, max(2, prep$true_k + 2))

  ok <- tryCatch({
    evaluate_clustering_grid(
      X = prep$X_sparsified,
      labels = prep$labels,
      dataset_name = nm,
      representation_name = "sparsified",
      k_values = k_values
    )
    TRUE
  }, error = function(e) {
    cat(sprintf("FAILED on %s\n", nm))
    cat(sprintf("MESSAGE: %s\n", e$message))
    FALSE
  })

  if (!ok) break
}



TRYING DATASET: hcv
Running dataset=hcv | representation=sparsified | method=kmeans | k=2 | n=615 | p=46
Running dataset=hcv | representation=sparsified | method=kmeans | k=3 | n=615 | p=46
Running dataset=hcv | representation=sparsified | method=kmeans | k=4 | n=615 | p=46
Running dataset=hcv | representation=sparsified | method=kmeans | k=5 | n=615 | p=46
Running dataset=hcv | representation=sparsified | method=kmeans | k=6 | n=615 | p=46
Running dataset=hcv | representation=sparsified | method=kmeans | k=7 | n=615 | p=46

TRYING DATASET: statlog_image_segmentation
Running dataset=statlog_image_segmentation | representation=sparsified | method=kmeans | k=2 | n=2310 | p=66
Running dataset=statlog_image_segmentation | representation=sparsified | method=kmeans | k=3 | n=2310 | p=66
Running dataset=statlog_image_segmentation | representation=sparsified | method=kmeans | k=4 | n=2310 | p=66
Running dataset=statlog_image_segmentation | representation=sparsified | method=kmeans | k=5 | n=2

## 12. Evaluate clustering before and after sparsification

For each dataset, the notebook evaluates:
- k-means
- hierarchical clustering
- Gaussian mixture clustering

It saves both the full grid of results and compact summaries.


In [15]:
# Evaluate UCI clustering before and after sparse orthogonalization (k-means only) -

uci_results_all <- data.frame()
uci_assignments_selected <- data.frame()

for (nm in names(uci_prepared)) {
  prep <- uci_prepared[[nm]]
  true_k <- prep$true_k
  k_values <- 2:min(8, max(3, true_k + 2), prep$n - 1)

  eval_original <- evaluate_clustering_grid(
    X = prep$X_original,
    labels = prep$labels,
    dataset_name = nm,
    representation_name = "original",
    k_values = k_values
  )

  eval_sparse <- evaluate_clustering_grid(
    X = prep$X_sparsified,
    labels = prep$labels,
    dataset_name = nm,
    representation_name = "sparsified",
    k_values = k_values
  )

  uci_results_all <- rbind(uci_results_all, eval_original$results, eval_sparse$results)

  for (rep_name in c("original", "sparsified")) {
    eval_obj <- if (rep_name == "original") eval_original else eval_sparse
    for (method_name in unique(eval_obj$results$method)) {
      key_name <- paste0(method_name, "_k", true_k)
      if (key_name %in% names(eval_obj$assignments)) {
        assign_df <- eval_obj$assignments[[key_name]]
        assign_df$dataset <- nm
        assign_df$representation <- rep_name
        assign_df$method <- method_name
        uci_assignments_selected <- rbind(uci_assignments_selected, assign_df)
      }
    }
  }
}

uci_best_silhouette <- do.call(
  rbind,
  lapply(split(uci_results_all, list(uci_results_all$dataset, uci_results_all$representation, uci_results_all$method), drop = TRUE), function(df) {
    df[which.max(df$silhouette), c("dataset", "representation", "method", "k", "silhouette", "davies_bouldin", "calinski_harabasz", "runtime_sec", "n_used")]
  })
)
row.names(uci_best_silhouette) <- NULL

uci_truek_metrics <- data.frame()
for (nm in names(uci_prepared)) {
  tk <- uci_prepared[[nm]]$true_k
  sub <- uci_results_all[uci_results_all$dataset == nm & uci_results_all$k == tk, ]
  uci_truek_metrics <- rbind(uci_truek_metrics, sub)
}

uci_truek_ari <- uci_truek_metrics[, c("dataset", "representation", "method", "k", "adjusted_rand_index")]
uci_truek_accuracy <- uci_truek_metrics[, c("dataset", "representation", "method", "k", "clustering_accuracy")]
uci_truek_purity <- uci_truek_metrics[, c("dataset", "representation", "method", "k", "purity")]

write.csv(uci_results_all, "outputs/uci_clustering_grid_metrics.csv", row.names = FALSE)
write.csv(uci_best_silhouette, "outputs/uci_clustering_best_silhouette.csv", row.names = FALSE)
write.csv(uci_truek_metrics, "outputs/uci_clustering_true_k_metrics.csv", row.names = FALSE)
write.csv(uci_truek_ari, "outputs/uci_clustering_ari_true_k.csv", row.names = FALSE)
write.csv(uci_truek_accuracy, "outputs/uci_clustering_accuracy_true_k.csv", row.names = FALSE)
write.csv(uci_truek_purity, "outputs/uci_clustering_purity_true_k.csv", row.names = FALSE)
write.csv(uci_assignments_selected, "outputs/uci_clustering_assignments_true_k.csv", row.names = FALSE)

capture.output(
  list(
    dataset_overview = uci_overview,
    best_silhouette = uci_best_silhouette,
    true_k_metrics = uci_truek_metrics,
    true_k_ari = uci_truek_ari,
    true_k_accuracy = uci_truek_accuracy,
    true_k_purity = uci_truek_purity
  ),
  file = "outputs/uci_clustering_summary.txt"
)

uci_best_silhouette
uci_truek_ari
uci_truek_accuracy
uci_truek_purity


Running dataset=breast_tissue | representation=original | method=kmeans | k=2 | n=106 | p=9
Running dataset=breast_tissue | representation=original | method=kmeans | k=3 | n=106 | p=9
Running dataset=breast_tissue | representation=original | method=kmeans | k=4 | n=106 | p=9
Running dataset=breast_tissue | representation=original | method=kmeans | k=5 | n=106 | p=9
Running dataset=breast_tissue | representation=original | method=kmeans | k=6 | n=106 | p=9
Running dataset=breast_tissue | representation=original | method=kmeans | k=7 | n=106 | p=9
Running dataset=breast_tissue | representation=original | method=kmeans | k=8 | n=106 | p=9
Running dataset=breast_tissue | representation=sparsified | method=kmeans | k=2 | n=106 | p=36
Running dataset=breast_tissue | representation=sparsified | method=kmeans | k=3 | n=106 | p=36
Running dataset=breast_tissue | representation=sparsified | method=kmeans | k=4 | n=106 | p=36
Running dataset=breast_tissue | representation=sparsified | method=kmea

dataset,representation,method,k,silhouette,davies_bouldin,calinski_harabasz,runtime_sec,n_used
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
banknote_authentication,original,kmeans,2,0.33375771,1.1996917,807.22021,0.04,1372
breast_tissue,original,kmeans,2,0.49255995,0.9575328,79.63129,0.00,106
hcv,original,kmeans,4,0.26024753,1.3634406,109.05223,0.02,615
libras_movement,original,kmeans,8,0.23444033,1.4388878,59.44990,0.08,360
pen_recognition_handwritten_digits,original,kmeans,8,0.28291924,1.2998000,2566.94748,1.32,10992
statlog_image_segmentation,original,kmeans,7,0.35430457,1.0809454,850.93351,0.19,2310
user_knowledge_modeling,original,kmeans,7,0.18682819,1.3957690,73.05415,0.03,403
vehicle_silhouettes,original,kmeans,2,0.38960414,1.0320468,588.94251,0.03,846
vertebral_column_2classes,original,kmeans,2,0.36290178,1.1308611,189.33892,0.02,310


,dataset,representation,method,k,adjusted_rand_index
,<chr>,<chr>,<chr>,<int>,<dbl>
14,breast_tissue,original,kmeans,6,0.28508012
141,breast_tissue,sparsified,kmeans,6,0.36085264
132,hcv,original,kmeans,5,0.07299919
133,hcv,sparsified,kmeans,5,0.05370325
114,wholesale_customers,original,kmeans,3,-0.01028161
116,wholesale_customers,sparsified,kmeans,3,-0.01009693
1211,vehicle_silhouettes,original,kmeans,4,0.07612445
1212,vehicle_silhouettes,sparsified,kmeans,4,0.12015079
158,statlog_image_segmentation,original,kmeans,7,0.45998203


,dataset,representation,method,k,clustering_accuracy
,<chr>,<chr>,<chr>,<int>,<dbl>
14,breast_tissue,original,kmeans,6,0.4811321
141,breast_tissue,sparsified,kmeans,6,0.5283019
132,hcv,original,kmeans,5,0.4032520
133,hcv,sparsified,kmeans,5,0.3252033
114,wholesale_customers,original,kmeans,3,0.5181818
116,wholesale_customers,sparsified,kmeans,3,0.3909091
1211,vehicle_silhouettes,original,kmeans,4,0.3593381
1212,vehicle_silhouettes,sparsified,kmeans,4,0.4491726
158,statlog_image_segmentation,original,kmeans,7,0.5450216


,dataset,representation,method,k,purity
,<chr>,<chr>,<chr>,<int>,<dbl>
14,breast_tissue,original,kmeans,6,0.5566038
141,breast_tissue,sparsified,kmeans,6,0.5849057
132,hcv,original,kmeans,5,0.8991870
133,hcv,sparsified,kmeans,5,0.8991870
114,wholesale_customers,original,kmeans,3,0.7181818
116,wholesale_customers,sparsified,kmeans,3,0.7181818
1211,vehicle_silhouettes,original,kmeans,4,0.3676123
1212,vehicle_silhouettes,sparsified,kmeans,4,0.4491726
158,statlog_image_segmentation,original,kmeans,7,0.5536797


## 13. Save retained clustering figures as PDF (300 dpi)


In [24]:
# Save selected clustering figures -------------------------------------------

save_plot_pdf(
  "figure10_uci_ari_at_true_k.pdf",
  plot_metric_pairs(
    uci_truek_metrics,
    value_col = "adjusted_rand_index",
    main_title = "ARI at the label-based number of clusters",
    xlab = "Adjusted Rand Index",
    higher_is_better = TRUE
  ),
  width = 9,
  height = 5
)

save_plot_pdf(
  "figure11_uci_accuracy_at_true_k.pdf",
  plot_metric_pairs(
    uci_truek_metrics,
    value_col = "clustering_accuracy",
    main_title = "Accuracy at the label-based number of clusters",
    xlab = "Clustering accuracy",
    higher_is_better = TRUE
  ),
  width = 9,
  height = 5
)

save_plot_pdf(
  "figure12_uci_purity_at_true_k.pdf",
  plot_metric_pairs_fixed_xlim(
    uci_truek_metrics,
    value_col = "purity",
    main_title = "Purity at the label-based number of clusters",
    xlab = "Purity",
    xlim_fixed = c(0.30, 1.00)
  ),
  width = 9,
  height = 5
)

cat("Selected UCI clustering PDFs saved (k-means only, sparse orthogonalized comparison).\n")


Selected UCI clustering PDFs saved (k-means only, sparse orthogonalized comparison).


## 14. Exploratory clustering on the Chinese housing dataset

This is an **exploratory unsupervised illustration only**.  
It clusters the years using the macroeconomic predictors `x1`–`x7`, without using the housing-price response `y`.


In [17]:
# Exploratory clustering on the Chinese housing dataset (sparse orthogonalized) --

housing_file <- find_data_file_multi(c("regression_data_1997_2023"))
if (is.na(housing_file)) stop("regression_data_1997_2023 file not found in data/ or current directory.")
housing_df <- read.csv(housing_file, stringsAsFactors = FALSE)
if (!"year" %in% names(housing_df)) {
  housing_df$year <- 1997:2023
}

housing_X <- as.matrix(housing_df[, c("x1", "x2", "x3", "x4", "x5", "x6", "x7")])
housing_X_scaled <- scale(housing_X)
housing_X_sparse <- sparse_orthogonalize_matrix(housing_X_scaled, n_blocks = SPARSE_ORTHO_BINS)

housing_eval_original <- evaluate_clustering_grid(
  X = housing_X_scaled,
  labels = NULL,
  dataset_name = "housing_years",
  representation_name = "original",
  k_values = 2:5,
  methods = c("kmeans")
)

housing_eval_sparse <- evaluate_clustering_grid(
  X = housing_X_sparse,
  labels = NULL,
  dataset_name = "housing_years",
  representation_name = "sparsified",
  k_values = 2:5,
  methods = c("kmeans")
)

housing_metrics <- rbind(housing_eval_original$results, housing_eval_sparse$results)
write.csv(housing_metrics, "outputs/housing_clustering_internal_metrics.csv", row.names = FALSE)

housing_best <- do.call(
  rbind,
  lapply(split(housing_metrics, list(housing_metrics$representation, housing_metrics$method), drop = TRUE), function(df) {
    df[which.max(df$silhouette), c("dataset", "representation", "method", "k", "silhouette", "davies_bouldin", "calinski_harabasz", "runtime_sec")]
  })
)
row.names(housing_best) <- NULL
write.csv(housing_best, "outputs/housing_clustering_best_models.csv", row.names = FALSE)

get_best_assignment <- function(eval_obj, best_df, representation_name, method_name) {
  sub <- best_df[best_df$representation == representation_name & best_df$method == method_name, ]
  if (nrow(sub) == 0) return(NULL)
  key <- paste0(method_name, "_k", sub$k[1])
  eval_obj$assignments[[key]]
}

housing_assignments <- data.frame(year = housing_df$year)

for (method_name in c("kmeans")) {
  assign_orig <- get_best_assignment(housing_eval_original, housing_best, "original", method_name)
  assign_sparse <- get_best_assignment(housing_eval_sparse, housing_best, "sparsified", method_name)

  if (!is.null(assign_orig)) {
    tmp <- rep(NA_integer_, nrow(housing_df))
    tmp[assign_orig$obs_index] <- assign_orig$predicted_cluster
    housing_assignments[[paste0("original_", method_name)]] <- tmp
  }

  if (!is.null(assign_sparse)) {
    tmp <- rep(NA_integer_, nrow(housing_df))
    tmp[assign_sparse$obs_index] <- assign_sparse$predicted_cluster
    housing_assignments[[paste0("sparsified_", method_name)]] <- tmp
  }
}

write.csv(housing_assignments, "outputs/housing_clustering_assignments.csv", row.names = FALSE)


plot_housing_internal_metrics <- function(df) {
  sub <- df[df$method == "kmeans", ]
  orig <- sub[sub$representation == "original", ]
  sprs <- sub[sub$representation == "sparsified", ]

  op <- par(mar = c(4, 4, 3, 2))
  on.exit(par(op), add = TRUE)

  plot(orig$k, orig$silhouette, type = "b", pch = 16, col = "black",
       ylim = range(sub$silhouette, na.rm = TRUE),
       xlab = "Number of clusters (k)",
       ylab = "Average silhouette width",
       main = "Chinese housing data - kmeans")
  lines(sprs$k, sprs$silhouette, type = "b", pch = 17, col = "red")
  legend("bottomright", legend = c("Original", "Sparsified"),
         pch = c(16, 17), col = c("black", "red"), bty = "n")
}

plot_housing_pca_clusters <- function(X1, cl1, X2, cl2, years) {
  op <- par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
  on.exit(par(op), add = TRUE)

  p1 <- prcomp(X1)
  s1 <- p1$x[, 1:2, drop = FALSE]
  plot(s1[, 1], s1[, 2], col = cl1, pch = 16,
       xlab = "PC1", ylab = "PC2", main = "Original data")
  text(s1[, 1], s1[, 2], labels = years, pos = 3, cex = 0.7)

  p2 <- prcomp(X2)
  s2 <- p2$x[, 1:2, drop = FALSE]
  plot(s2[, 1], s2[, 2], col = cl2, pch = 16,
       xlab = "PC1", ylab = "PC2", main = "Sparsified data")
  text(s2[, 1], s2[, 2], labels = years, pos = 3, cex = 0.7)
}

save_plot_pdf(
  "figure13_housing_internal_metrics.pdf",
  plot_housing_internal_metrics(housing_metrics),
  width = 8,
  height = 7
)

best_orig_kmeans <- housing_best$k[housing_best$representation == "original" & housing_best$method == "kmeans"][1]
best_sparse_kmeans <- housing_best$k[housing_best$representation == "sparsified" & housing_best$method == "kmeans"][1]

orig_clusters <- housing_eval_original$assignments[[paste0("kmeans_k", best_orig_kmeans)]]$predicted_cluster
sprs_clusters <- housing_eval_sparse$assignments[[paste0("kmeans_k", best_sparse_kmeans)]]$predicted_cluster

save_plot_pdf(
  "figure14_housing_pca_clusters.pdf",
  plot_housing_pca_clusters(housing_X_scaled, orig_clusters, housing_X_sparse, sprs_clusters, housing_df$year),
  width = 10,
  height = 5
)

capture.output(
  list(
    housing_best_models = housing_best,
    housing_metrics = housing_metrics
  ),
  file = "outputs/housing_clustering_summary.txt"
)

housing_best
housing_assignments


Running dataset=housing_years | representation=original | method=kmeans | k=2 | n=27 | p=7
Running dataset=housing_years | representation=original | method=kmeans | k=3 | n=27 | p=7
Running dataset=housing_years | representation=original | method=kmeans | k=4 | n=27 | p=7
Running dataset=housing_years | representation=original | method=kmeans | k=5 | n=27 | p=7
Running dataset=housing_years | representation=sparsified | method=kmeans | k=2 | n=27 | p=28
Running dataset=housing_years | representation=sparsified | method=kmeans | k=3 | n=27 | p=28
Running dataset=housing_years | representation=sparsified | method=kmeans | k=4 | n=27 | p=28
Running dataset=housing_years | representation=sparsified | method=kmeans | k=5 | n=27 | p=28


dataset,representation,method,k,silhouette,davies_bouldin,calinski_harabasz,runtime_sec
<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<dbl>,<dbl>
housing_years,original,kmeans,2,0.6565444,0.4446674,95.22438,0
housing_years,sparsified,kmeans,4,0.6759242,0.5505812,39.98530,0


year,original_kmeans,sparsified_kmeans
<int>,<int>,<int>
1997,2,2
1998,2,2
1999,2,2
2000,2,2
2001,2,2
2002,2,2
2003,2,2
2004,2,3
2005,2,3


## 15. Files created by the retained clustering section

### In `outputs/`
- `uci_dataset_overview.csv`
- `uci_clustering_grid_metrics.csv`
- `uci_clustering_best_silhouette.csv`
- `uci_clustering_true_k_metrics.csv`
- `uci_clustering_accuracy_true_k.csv`
- `uci_clustering_assignments_true_k.csv`
- `uci_clustering_summary.txt`
- `figure10_uci_best_silhouette.pdf`
- `figure11_uci_sparsity_summary.pdf`
- `figure12_uci_accuracy_at_true_k.pdf`
- `housing_clustering_internal_metrics.csv`
- `housing_clustering_best_models.csv`
- `housing_clustering_assignments.csv`
- `housing_clustering_summary.txt`
- `figure13_housing_internal_metrics.pdf`
- `figure14_housing_pca_clusters.pdf`
